In [1]:
import numpy as np
import pymupdf 
import torch
import random



random_seed = 42



In [41]:
import pymupdf
from pathlib import Path

# Folder containing The_Last_Wish.pdf
folder = Path("The_Last_Wish.pdf").resolve().parent

# Find all PDF files in that folder
pdf_files = sorted(folder.glob("*.pdf"))

text = ""

for pdf_file in pdf_files:
    print(f"Reading: {pdf_file.name}")

    with pymupdf.open(pdf_file) as doc:
        for page in doc:
            text += page.get_text() + "\n"

print(f"\nTotal PDFs: {len(pdf_files)}")
print(f"Total characters: {len(text):,}")

Reading: 2-The Sword of Destiny.pdf
Reading: 3-Blood of Elves.pdf
Reading: 4-The Time of Contempt.pdf
Reading: 5-Baptism of Fire.pdf
Reading: 6-The Tower of Swallows.pdf
Reading: 7-Lady of the Lake.pdf
Reading: 8-Season of Storms.pdf
Reading: The_Last_Wish.pdf

Total PDFs: 8
Total characters: 5,548,429


In [42]:
length = len(text)
length

5548429

In [43]:
# Create character replacement mapping
replacements = str.maketrans({
    'ç': 'c',
    'é': 'e',
    'ê': 'e',
    'ë': 'e',
    'ï': 'i',
    'ż': 'z',
    '\u200a': ' '
})

# Apply replacements to the text
text = text.translate(replacements)

# Rebuild vocabulary
chars = sorted(set(text))
vocab_size = len(chars)

# Display results
print("Total characters:", len(text))
print("Vocabulary size:", vocab_size)
print("Characters:", chars)

Total characters: 5548429
Vocabulary size: 102
Characters: ['\n', ' ', '!', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\xa0', '©', 'Á', 'à', 'á', 'â', 'ä', 'è', 'í', 'î', 'ó', 'ô', 'ö', 'ú', 'ę', 'ł', 'ᓼ', '\u200b', '–', '—', '‘', '’', '“', '”', '…']


In [44]:
unique_chars = sorted(set(text))
unique_chars_count = len(unique_chars)
print(f"Number of unique characters: {unique_chars_count}, Unique characters: {unique_chars}    ")

Number of unique characters: 102, Unique characters: ['\n', ' ', '!', "'", '(', ')', '*', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '\xa0', '©', 'Á', 'à', 'á', 'â', 'ä', 'è', 'í', 'î', 'ó', 'ô', 'ö', 'ú', 'ę', 'ł', 'ᓼ', '\u200b', '–', '—', '‘', '’', '“', '”', '…']    


In [45]:
from collections import Counter

char_counts = Counter(text)

# Print characters from most frequent to least frequent
for char, count in char_counts.most_common()[::-1]:
    print(f"{repr(char):10} : {count}")

'â'        : 1
'Á'        : 1
'ä'        : 1
'ę'        : 1
'î'        : 2
'ó'        : 2
'è'        : 3
'ô'        : 3
'\u200b'   : 4
'ᓼ'        : 6
'ł'        : 7
'@'        : 7
'ú'        : 8
'à'        : 8
'í'        : 9
'ö'        : 14
'/'        : 15
'\xa0'     : 20
'©'        : 20
'5'        : 21
'8'        : 31
'4'        : 34
'7'        : 34
'6'        : 41
'9'        : 49
'3'        : 50
')'        : 51
'('        : 51
'2'        : 64
'á'        : 70
'0'        : 77
'X'        : 84
'1'        : 119
'Q'        : 297
'Z'        : 505
'U'        : 639
':'        : 648
'J'        : 789
"'"        : 792
';'        : 1031
'—'        : 1395
'*'        : 1540
'K'        : 1800
'–'        : 2052
'V'        : 2181
'P'        : 2800
'z'        : 3198
'R'        : 3270
'L'        : 3274
'O'        : 3371
'E'        : 3712
'q'        : 3724
'F'        : 3798
'-'        : 3842
'j'        : 3947
'M'        : 4467
'N'        : 5043
'…'        : 5148
'x'        : 5297
'D'        : 5811
'G'   

In [46]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}    
encoding = lambda s: [stoi[c] for c in s]  # encoder: take a string, output a list of integers
decoding = lambda l: ''.join([itos[i] for i in l])  # decoder: take a list of integers, output a string

In [47]:
data = torch.tensor(encoding(text), dtype=torch.long)
data[:1000]  # first 1000 characters as integers
data.shape


torch.Size([5548429])

In [48]:
train_size = int(0.9 * len(data))
train_data = data[:train_size]
val_data = data[train_size:]

In [49]:
batch_size = 64  # how many independent sequences will we process in parallel?
block_size = 256  # what is the maximum context length for predictions?


# Set random seed once
torch.manual_seed(random_seed)

# Select device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)


def get_batch(split):
    # Select training or validation dataset
    data = train_data if split == 'train' else val_data

    # Generate 64 random starting positions
    idx = torch.randint(
        len(data) - block_size,
        (batch_size,)
    )

    # Input sequences: shape (64, 256)
    x = torch.stack([
        data[i:i + block_size]
        for i in idx
    ])

    # Target sequences shifted by one position
    y = torch.stack([
        data[i + 1:i + block_size + 1]
        for i in idx
    ])

    # Move tensors to GPU if available
    x = x.to(device)
    y = y.to(device)

    return x, y


# Generate a training batch
xb, yb = get_batch('train')

print("Input shape:", xb.shape)
print("Target shape:", yb.shape)

print("x[0]:", xb[0])
print("y[0]:", yb[0])

Using device: cuda
Input shape: torch.Size([64, 256])
Target shape: torch.Size([64, 256])
x[0]: tensor([100,   1,  69,  58,  55,   1,  69,  51,  59,  54,   1,  64,  51,  70,
         71,  68,  51,  62,  62,  75,   9,   1,  99,  44,  58,  55,  68,  55,
          1,  51,  68,  55,   1,  51,   1,  56,  55,  73,   0,  62,  59,  70,
         70,  62,  55,   1,  70,  58,  59,  64,  57,  69,   1,  59,  64,   1,
         70,  58,  55,   1,  69,  51,  54,  54,  62,  55,   8,  52,  51,  57,
         69,   1,  73,  58,  59,  53,  58,   1,  33,  98,  62,  62,   1,  64,
         55,  55,  54,   9, 100,   0,  99,  49,  65,  71,   1,  57,  51,  72,
         55,   1,  63,  55,   1,  51,   1,  72,  55,  68,  75,   1,  54,  59,
         69,  51,  57,  68,  55,  55,  51,  52,  62,  55,   1,  55,  74,  66,
         55,  68,  59,  55,  64,  53,  55,   1,  64,  65,  70,   1,  69,  65,
          1,  62,  65,  64,  57,   0,  51,  57,  65,   7, 100,   1,  58,  55,
          1,  63,  71,  70,  70,  55,  68,  55

In [50]:
#create embedding table
n_embeddings = vocab_size  # number of unique characters
embedding_dim = 64  # size of each embedding vector
embedding_table = torch.nn.Embedding(n_embeddings, embedding_dim).to(device)
embedded_xb = embedding_table(xb)
embedded_xb.shape  # should be (64, 256, 32)


torch.Size([64, 256, 64])

In [51]:
embedded_xb[0,0]

tensor([ 0.4998,  0.9064, -0.2733, -0.8445,  0.0674, -1.1771, -2.3193,  1.0882,
         1.7532, -1.3527, -1.7171,  0.5395, -0.4648,  0.1551, -0.9332,  2.0512,
        -0.0040,  2.6462,  1.7450,  0.2889,  1.1111,  1.1171, -0.7967,  1.0229,
        -1.0742, -1.4499, -0.5232,  0.3438,  0.2038, -1.4714,  0.1978, -0.3223,
         1.8926,  0.6904,  1.4353,  1.0865, -0.3989, -2.0487, -1.1373,  0.9046,
         0.5280, -0.5005,  0.0379, -1.0130,  0.7844,  1.3281, -0.5139, -0.0483,
        -0.2073,  0.4188,  1.7955, -0.6994, -0.4368,  0.3443, -0.5315, -1.7879,
         0.5760, -0.3711, -0.0215, -0.3246,  0.7542, -1.7302, -0.6405, -0.0244],
       device='cuda:0', grad_fn=<SelectBackward0>)

In [52]:
dot_id = stoi['.']
dot_embedding = embedding_table.weight[dot_id]

print("Token ID:", dot_id)
print("Embedding:", dot_embedding)
print("Shape:", dot_embedding.shape)

Token ID: 9
Embedding: tensor([-1.2787, -0.0384,  1.9138,  0.3378,  0.1251, -0.7622, -1.1906,  0.7756,
         0.4557,  0.2503, -1.3611,  1.8018, -0.0743, -0.1566, -0.8708, -0.6411,
        -0.4146, -0.6902, -0.2300, -2.1723,  0.0877,  1.0938, -0.1177, -0.2986,
        -0.9536, -0.0925, -1.0167, -0.0077, -0.5182,  0.8395,  0.0585, -1.6682,
         2.1296, -1.5181,  0.1387, -1.1798, -0.5297,  0.9625,  0.2794, -0.5718,
        -2.7936, -0.7112,  0.5235, -1.7106,  0.8385, -0.2698,  0.1231,  0.8758,
         0.1513,  0.7394,  0.2731,  2.7312,  0.4320, -0.3092, -0.0966,  1.5419,
        -0.1087, -0.4189,  1.4384, -0.7068, -1.2520,  3.0250,  1.3463,  0.8556],
       device='cuda:0', grad_fn=<SelectBackward0>)
Shape: torch.Size([64])


In [53]:
position_emb = torch.nn.Embedding(block_size, embedding_dim).to(device)

numbers = torch.arange(block_size).to(device)
pos = position_emb(numbers)

In [54]:
pos[0] + dot_embedding

tensor([ 1.3119, -0.1826,  1.4369,  0.1089,  0.4955,  0.2621, -0.8396,  0.6596,
        -0.0572,  1.1117,  1.2567,  2.6541,  0.2700, -0.3761,  0.8244,  0.0902,
        -1.1241,  0.8203,  0.6844, -3.4673,  1.0383, -0.2478,  0.8407, -2.1331,
        -1.5481, -1.7635, -2.3609, -0.7999, -0.3929, -0.1577,  0.9902, -2.7672,
         3.0113, -3.5506, -0.2135, -1.0791,  0.9958, -0.6590,  0.8093, -1.9437,
        -3.4234, -1.3678,  0.9479, -2.1118,  0.3495,  0.6873, -0.7206,  0.5077,
         0.5684, -0.1319,  0.1109,  1.7831, -0.2215, -0.6677, -0.9141,  1.9155,
         0.8035,  0.2592,  1.4943, -2.0463, -1.5521,  3.1850,  0.6693,  1.6061],
       device='cuda:0', grad_fn=<AddBackward0>)

In [55]:
x_b = embedded_xb + pos
x_b[0,0]

tensor([ 3.0903,  0.7622, -0.7503, -1.0734,  0.4379, -0.1528, -1.9684,  0.9721,
         1.2403, -0.4913,  0.9007,  1.3917, -0.1205, -0.0643,  0.7620,  2.7824,
        -0.7136,  4.1568,  2.6593, -1.0061,  2.0617, -0.2244,  0.1617, -0.8115,
        -1.6688, -3.1209, -1.8674, -0.4484,  0.3291, -2.4686,  1.1295, -1.4213,
         2.7743, -1.3421,  1.0830,  1.1871,  1.1266, -3.6702, -0.6075, -0.4672,
        -0.1018, -1.1572,  0.4623, -1.4142,  0.2954,  2.2852, -1.3575, -0.4164,
         0.2097, -0.4525,  1.6333, -1.6475, -1.0903, -0.0142, -1.3491, -1.4143,
         1.4882,  0.3071,  0.0344, -1.6641,  0.4540, -1.5702, -1.3174,  0.7261],
       device='cuda:0', grad_fn=<SelectBackward0>)

In [56]:
x_b.shape

torch.Size([64, 256, 64])

In [57]:
kl = torch.tensor([[[1, 1], [1, 1], [1, 1], [1, 1]],
                   [[1, 1], [1, 1], [1, 1], [1, 1]],
                   [[1, 1], [1, 1], [1, 1], [1, 1]]])
print( kl.shape)
lk = torch.tensor([[2,3],
                  [2,4],
                  [2,5],
                  [2,6]])

print(lk.shape)
kkl = kl + lk
print(kkl)

torch.Size([3, 4, 2])
torch.Size([4, 2])
tensor([[[3, 4],
         [3, 5],
         [3, 6],
         [3, 7]],

        [[3, 4],
         [3, 5],
         [3, 6],
         [3, 7]],

        [[3, 4],
         [3, 5],
         [3, 6],
         [3, 7]]])


In [58]:
qw = torch.nn.Linear(embedding_dim, 64, bias=False).to(device)
kw = torch.nn.Linear(embedding_dim, 64, bias=False).to(device)
vw = torch.nn.Linear(embedding_dim, 64, bias=False).to(device)

q = qw(x_b)
k = kw(x_b)
v = vw(x_b)

# Attention scores
r = (q @ k.transpose(-2, -1)) / (64 ** 0.5)

# Causal mask
tril = torch.tril(torch.ones(block_size, block_size, device=device))
r = r.masked_fill(tril == 0, float('-inf'))

# Attention probabilities
soft = torch.nn.functional.softmax(r, dim=-1)

# Weighted sum of values
v = soft @ v

print(v.shape)
# torch.Size([64, 256, 64])

torch.Size([64, 256, 64])


In [59]:
res = torch.nn.Linear(64, embedding_dim).to(device)
out = res(v)
x = x_b + out
x.shape


torch.Size([64, 256, 64])

In [60]:
norm = torch.nn.LayerNorm(embedding_dim).to(device)
x = norm(x)
x.shape

torch.Size([64, 256, 64])

In [61]:
ffn1 = torch.nn.Linear(embedding_dim,128).to(device)
ff = ffn1(x)
ff = torch.relu(ff)
ffn2 = torch.nn.Linear(128,embedding_dim).to(device)
fff = ffn2(ff)
ffff = fff + x
norm1 = torch.nn.LayerNorm(embedding_dim).to(device)

xfn = norm1(ffff)
xfn.shape



torch.Size([64, 256, 64])

In [62]:
fl = torch.nn.Linear(embedding_dim,vocab_size).to(device)
logit = fl(xfn)
logit.shape

torch.Size([64, 256, 102])

In [63]:
import torch.nn.functional as F

B, T, V = logit.shape

logits_flat = logit.reshape(B * T, V)
targets_flat = yb.reshape(B * T)

loss = F.cross_entropy(logits_flat, targets_flat)

print(loss.item())

4.80577278137207


In [64]:
layers = [
    embedding_table,
    position_emb,
    qw, kw, vw,
    res,
    norm,
    ffn1, ffn2,
    norm1,
    fl
]

parameters = [
    p
    for layer in layers
    for p in layer.parameters()
]

optimizer = torch.optim.AdamW(parameters, lr=0.001)

In [65]:
optimizer.zero_grad()
loss.backward()
optimizer.step()

In [66]:

def forward(xb):
    # 1. Token + positional embeddings
    token_emb = embedding_table(xb)
    positions = torch.arange(xb.shape[1], device=device)
    pos_emb = position_emb(positions)

    x_b = token_emb + pos_emb

    # 2. Q, K, V
    q = qw(x_b)
    k = kw(x_b)
    v = vw(x_b)

    # 3. Scaled dot-product attention
    r = (q @ k.transpose(-2, -1)) / (64 ** 0.5)

    # 4. Causal masking
    T = xb.shape[1]
    tril = torch.tril(torch.ones(T, T, device=device))
    r = r.masked_fill(tril == 0, float('-inf'))

    # 5. Softmax and weighted values
    soft = F.softmax(r, dim=-1)
    v = soft @ v

    # 6. Output projection + residual + normalization
    out = res(v)
    x = norm(x_b + out)

    # 7. Feed-forward network
    ff = ffn1(x)
    ff = torch.relu(ff)
    ff = ffn2(ff)

    # 8. Residual + second normalization
    xfn = norm1(x + ff)

    # 9. Final linear layer
    logits = fl(xfn)

    return logits

In [67]:
max_iters = 15000

for step in range(max_iters):

    # Fresh batch
    xb, yb = get_batch('train')

    # Forward pass
    logits = forward(xb)

    # Calculate loss
    B, T, V = logits.shape

    loss = F.cross_entropy(
        logits.reshape(B * T, V),
        yb.reshape(B * T)
    )

    # Backpropagation
    optimizer.zero_grad()
    loss.backward()

    # Update weights
    optimizer.step()

    # Print progress
    if step % 100 == 0:
        print(f"Step {step}: Loss = {loss.item():.4f}")

Step 0: Loss = 4.7401


Step 100: Loss = 2.7099
Step 200: Loss = 2.6040
Step 300: Loss = 2.5765
Step 400: Loss = 2.5460
Step 500: Loss = 2.5489
Step 600: Loss = 2.5225
Step 700: Loss = 2.4945
Step 800: Loss = 2.4715
Step 900: Loss = 2.3621
Step 1000: Loss = 2.3018
Step 1100: Loss = 2.2141
Step 1200: Loss = 2.1995
Step 1300: Loss = 2.1549
Step 1400: Loss = 2.1533
Step 1500: Loss = 2.1493
Step 1600: Loss = 2.1385
Step 1700: Loss = 2.1381
Step 1800: Loss = 2.1384
Step 1900: Loss = 2.0837
Step 2000: Loss = 2.1342
Step 2100: Loss = 2.0992
Step 2200: Loss = 2.1061
Step 2300: Loss = 2.0941
Step 2400: Loss = 2.0800
Step 2500: Loss = 2.0805
Step 2600: Loss = 2.0890
Step 2700: Loss = 2.0632
Step 2800: Loss = 2.0547
Step 2900: Loss = 2.0783
Step 3000: Loss = 2.0586
Step 3100: Loss = 2.0526
Step 3200: Loss = 2.0497
Step 3300: Loss = 2.0483
Step 3400: Loss = 2.0248
Step 3500: Loss = 2.0466
Step 3600: Loss = 2.0375
Step 3700: Loss = 2.0497
Step 3800: Loss = 2.0372
Step 3900: Loss = 2.0464
Step 4000: Loss = 2.0333
Step 4100

In [68]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def estimate_loss(eval_iters=100):
    losses = {}

    for split in ['train', 'val']:
        total_loss = 0.0

        for _ in range(eval_iters):
            xb, yb = get_batch(split)

            logits = forward(xb)

            B, T, V = logits.shape

            loss = F.cross_entropy(
                logits.reshape(B * T, V),
                yb.reshape(B * T)
            )

            total_loss += loss.item()

        losses[split] = total_loss / eval_iters

    return losses


results = estimate_loss()

print(f"Training Loss:   {results['train']:.4f}")
print(f"Validation Loss: {results['val']:.4f}")

Training Loss:   1.8869
Validation Loss: 1.8994


In [69]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(prompt, max_new_tokens=300, temperature=1.0):

    # Convert characters into token IDs
    tokens = encoding(prompt)

    # Shape: (1, number_of_tokens)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):

        # Keep only the last 256 tokens
        idx_cond = idx[:, -block_size:]

        # Forward pass through our trained Transformer
        logits = forward(idx_cond)

        # Get predictions for the last token only
        logits = logits[:, -1, :]

        # Control randomness
        logits = logits / temperature

        # Convert logits into probabilities
        probs = F.softmax(logits, dim=-1)

        # Sample the next character
        next_token = torch.multinomial(probs, num_samples=1)

        # Append predicted character
        idx = torch.cat((idx, next_token), dim=1)

    # Convert token IDs back into text
    return decoding(idx[0].tolist())

In [70]:
prompt = "Geralt"

generated_text = generate(
    prompt,
    max_new_tokens=500,
    temperature=0.8
)

print(generated_text)

Geraltan.
‘You withem they coming prings throunds and in made at word.’
‘Forkether chaught by does. Did hear many? He
mards.’
‘She surry merst cruping bill in to you hathere yelf, to this he try,’ tim free bled. The and the borned. “a scriting talt mand hot of the supporcrass. A simples off, ning bucking shat whing and from tely, want a blatest conve would thatly and ther or and the Signate atred somed know. Ciri rejous. Bonhard with me. “Eldate sented hisgread, ming to story faced,” nere, with broked


In [78]:
#######
import torch
import torch.nn as nn
import torch.nn.functional as F

class TransformerBlock(nn.Module):
    def __init__(self):
        super().__init__()

        self.qw = nn.Linear(embedding_dim, 128, bias=False)
        self.kw = nn.Linear(embedding_dim,128)
        self.vw = nn.Linear(embedding_dim, 128, bias=False)

        self.res = nn.Linear(128, embedding_dim)
        self.norm = nn.LayerNorm(embedding_dim)

        self.ffn1 = nn.Linear(embedding_dim, 256)
        self.ffn2 = nn.Linear(256, embedding_dim)
        self.norm1 = nn.LayerNorm(embedding_dim)

    def forward(self, x):
        B, T, C = x.shape

        q = self.qw(x)
        k = self.kw(x)
        v = self.vw(x)

        r = (q @ k.transpose(-2, -1)) / (128 ** 0.5)

        tril = torch.tril(
            torch.ones(T, T, device=x.device, dtype=torch.bool)
        )
        r = r.masked_fill(~tril, float('-inf'))

        soft = F.softmax(r, dim=-1)
        attention_output = soft @ v

        # First residual connection and normalization
        x = self.norm(x + self.res(attention_output))

        # Feed-forward network
        ff = self.ffn1(x)
        ff = F.relu(ff)
        ff = self.ffn2(ff)

        # Second residual connection and normalization
        x = self.norm1(x + ff)

        return x

In [79]:
blocks = nn.Sequential(
    *[TransformerBlock() for _ in range(6)]
).to(device)

In [80]:
def forward(xb):
    B, T = xb.shape

    token_emb = embedding_table(xb)

    positions = torch.arange(T, device=device)
    pos_emb = position_emb(positions)

    x = token_emb + pos_emb

    # Pass through all six Transformer blocks
    x = blocks(x)

    # Predict vocabulary logits
    logits = fl(x)

    return logits

In [81]:
layers = [
    embedding_table,
    position_emb,
    blocks,
    fl
]

parameters = [
    p
    for layer in layers
    for p in layer.parameters()
]

optimizer = torch.optim.AdamW(parameters, lr=0.001)

print("Total parameters:", sum(p.numel() for p in parameters))

Total parameters: 427366


In [82]:
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch('train')

    logits = forward(xb)

    B, T, V = logits.shape

    loss = F.cross_entropy(
        logits.reshape(B * T, V),
        yb.reshape(B * T)
    )

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(f"Step {step}: Loss = {loss.item():.4f}")

Step 0: Loss = 4.9106
Step 500: Loss = 2.1071
Step 1000: Loss = 1.8166
Step 1500: Loss = 1.6783
Step 2000: Loss = 1.6103
Step 2500: Loss = 1.5634
Step 3000: Loss = 1.4970
Step 3500: Loss = 1.4673
Step 4000: Loss = 1.4723
Step 4500: Loss = 1.4580


In [83]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def generate(prompt, max_new_tokens=500, temperature=0.8):
    blocks.eval()

    idx = torch.tensor(
        [encoding(prompt)],
        dtype=torch.long,
        device=device
    )

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]

        logits = forward(idx_cond)
        logits = logits[:, -1, :] / temperature

        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, next_token), dim=1)

    return decoding(idx[0].tolist())


print(generate("Geralt", max_new_tokens=500))

Geralt could alone, or finds. “And I say the are simple in my tripper gractical – an
impride. There just immediately?”
“Do not be belowed it.” Geralt said his fingers with the saddle and strong of
people, stopped stupid a study towed for envousia. “What’s that come
thanks. Do you know arrived you got to can els, and
the see, in my ago, spirated of my there, and by they with senten than leaped to
constilly but the knige son armed the fate strength the blood of trees. There’s
upraces and the Spirrels Ri


In [84]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def estimate_loss(eval_iters=100):
    losses = {}

    blocks.eval()

    for split in ['train', 'val']:
        total_loss = 0.0

        for _ in range(eval_iters):
            xb, yb = get_batch(split)

            logits = forward(xb)

            B, T, V = logits.shape

            loss = F.cross_entropy(
                logits.reshape(B * T, V),
                yb.reshape(B * T)
            )

            total_loss += loss.item()

        losses[split] = total_loss / eval_iters

    blocks.train()
    return losses


results = estimate_loss()

print(f"Training Loss:   {results['train']:.4f}")
print(f"Validation Loss: {results['val']:.4f}")

Training Loss:   1.4358
Validation Loss: 1.5185
